## Example 1: Read an .xlsx workbook

In [ ]:
import importlib.metadata
import io

import xlsxwriter
from python_calamine import CalamineWorkbook

# Example 1: sheet names and rows of a workbook written by xlsxwriter
assert importlib.metadata.version("python-calamine") == "0.9.1"
buf = io.BytesIO()
with xlsxwriter.Workbook(buf) as wb:
    wb.add_worksheet("first").write_row(0, 0, ["a", 1, 2.5])
    wb.add_worksheet("second").write(0, 0, "b")
book = CalamineWorkbook.from_filelike(io.BytesIO(buf.getvalue()))
assert book.sheet_names == ["first", "second"], book.sheet_names
assert book.get_sheet_by_name("first").to_python() == [["a", 1.0, 2.5]]
print("Example 1 OK:", book.sheet_names)

## Example 2: Dates, booleans and empty cells

In [ ]:
import datetime
import io

import xlsxwriter
from python_calamine import CalamineWorkbook

# Example 2: Excel dates come back as Python datetimes, booleans as bool, gaps as ""
buf = io.BytesIO()
with xlsxwriter.Workbook(buf) as wb:
    ws = wb.add_worksheet()
    ws.write_datetime(0, 0, datetime.datetime(2026, 11, 7, 12, 30), wb.add_format({"num_format": "yyyy-mm-dd hh:mm"}))
    ws.write_boolean(0, 1, True)
    ws.write(0, 3, "after a gap")
row = CalamineWorkbook.from_filelike(io.BytesIO(buf.getvalue())).get_sheet_by_index(0).to_python()[0]
assert row == [datetime.datetime(2026, 11, 7, 12, 30), True, "", "after a gap"], row
print("Example 2 OK:", row)

## Example 3: Sheet metadata and partial reads

In [ ]:
import io

import xlsxwriter
from python_calamine import CalamineWorkbook

# Example 3: dimensions without reading every value, and reading only the first rows
buf = io.BytesIO()
with xlsxwriter.Workbook(buf) as wb:
    ws = wb.add_worksheet("big")
    for r in range(500):
        ws.write_row(r, 0, [r, r * 2, f"row {r}"])
sheet = CalamineWorkbook.from_filelike(io.BytesIO(buf.getvalue())).get_sheet_by_name("big")
assert (sheet.height, sheet.width) == (500, 3), (sheet.height, sheet.width)
head = sheet.to_python(nrows=3)
assert len(head) == 3 and head[2] == [2.0, 4.0, "row 2"], head
print("Example 3 OK:", head)

## Example 4: pandas.read_excel(engine='calamine')

In [ ]:
import io

import pandas as pd

# Example 4: pandas' fastest Excel reader is python-calamine
df = pd.DataFrame({"id": [1, 2, 3], "label": ["x", "y", "z"]})
buf = io.BytesIO()
df.to_excel(buf, index=False, engine="openpyxl")
back = pd.read_excel(io.BytesIO(buf.getvalue()), engine="calamine")
assert back.equals(df), back
print("Example 4 OK:\n", back)

## Example 5: An OpenDocument (.ods) spreadsheet

In [ ]:
import io
import zipfile

from python_calamine import CalamineWorkbook

# Example 5: calamine also reads LibreOffice's .ods; pandas has no .ods writer without odfpy, so it is built by hand
ns = ('xmlns:office="urn:oasis:names:tc:opendocument:xmlns:office:1.0" '
      'xmlns:table="urn:oasis:names:tc:opendocument:xmlns:table:1.0" xmlns:text="urn:oasis:names:tc:opendocument:xmlns:text:1.0"')
cell = lambda v: f'<table:table-cell office:value-type="float" office:value="{v}"><text:p>{v}</text:p></table:table-cell>'
rows = "".join(f"<table:table-row>{cell(i)}{cell(i * i)}</table:table-row>" for i in range(1, 4))
content = f'<office:document-content {ns}><office:body><office:spreadsheet><table:table table:name="sq">{rows}</table:table></office:spreadsheet></office:body></office:document-content>'
manifest = ('<manifest:manifest xmlns:manifest="urn:oasis:names:tc:opendocument:xmlns:manifest:1.0">'
            '<manifest:file-entry manifest:full-path="/" manifest:media-type="application/vnd.oasis.opendocument.spreadsheet"/></manifest:manifest>')
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.writestr("mimetype", "application/vnd.oasis.opendocument.spreadsheet")
    z.writestr("content.xml", content)
    z.writestr("META-INF/manifest.xml", manifest)
assert CalamineWorkbook.from_filelike(io.BytesIO(buf.getvalue())).get_sheet_by_name("sq").to_python() == [[1.0, 1.0], [2.0, 4.0], [3.0, 9.0]]
print("Example 5 OK")